# Notebook 11: Feature Engineering for ML Systems

## Why This Matters

Building features that work in a notebook is very different from building features that work reliably in a production ML system serving millions of requests per day. Feature stores, training-serving skew, temporal features, and entity-level aggregation are the engineering concerns that separate research prototypes from production systems — and they are prominently tested in ML system design interviews at companies like Meta, Google, Airbnb, and Stripe. This notebook covers the practical engineering disciplines: building time-aware features without leakage, handling entity-level aggregations efficiently, detecting and managing training-serving skew, and the principles behind feature store architecture.

## 1. Training-Serving Skew: The Silent Production Killer

**Training-serving skew** (also called train-serve skew) occurs when the features computed at training time differ from the features computed at serving time. It is the most common reason ML models degrade or fail silently in production.

Sources of skew:
1. **Different code paths**: Python pandas in training, Java/SQL in serving → different rounding, NA handling, sorting
2. **Different data windows**: training uses full historical data; serving uses a rolling window
3. **Race conditions**: serving computes features before all data for the current time window has arrived
4. **Schema drift**: upstream data sources add/rename/remove columns

The fix: define features in a **feature store** where a single feature definition is used for both training and serving. Major implementations: Feast (open-source), Tecton, Vertex AI Feature Store, Hopsworks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime, timedelta

# Simulate training-serving skew
# Scenario: mean encoding computed differently in training vs serving
np.random.seed(42)
n = 2000

df_train = pd.DataFrame({
    'user_id':   np.random.choice(range(100), n),
    'category':  np.random.choice(['A','B','C','D'], n, p=[0.4,0.3,0.2,0.1]),
    'amount':    np.random.exponential(50, n),
    'label':     np.random.binomial(1, 0.2, n)
})

# Training: compute category mean amount (correct, using full training set statistics)
cat_mean_train = df_train.groupby('category')['amount'].mean()
df_train['cat_mean_amount'] = df_train['category'].map(cat_mean_train)

# Serving: a bug causes category mean to be computed on a different sample (e.g., last 100 rows)
cat_mean_serve_buggy = df_train.tail(100).groupby('category')['amount'].mean()

# New serving-time data
df_serve = pd.DataFrame({
    'user_id':  np.random.choice(range(100), 200),
    'category': np.random.choice(['A','B','C','D'], 200, p=[0.4,0.3,0.2,0.1]),
    'amount':   np.random.exponential(50, 200),
})

df_serve['cat_mean_correct'] = df_serve['category'].map(cat_mean_train)  # correct
df_serve['cat_mean_buggy']   = df_serve['category'].map(cat_mean_serve_buggy)  # skewed

print("Category mean amounts — Training statistics:")
print(cat_mean_train.round(2))
print("\nCategory mean amounts — Serving (buggy, last 100 rows only):")
print(cat_mean_serve_buggy.round(2))
print("\nRelative error (%):", ((cat_mean_serve_buggy - cat_mean_train) / cat_mean_train * 100).round(1))

## 2. Temporal Feature Engineering

Time is one of the most information-rich feature dimensions in production systems. Correct temporal feature engineering requires two disciplines:

**Point-in-time correctness**: each training example can only use features computed from data available *before* the event timestamp. Violating this is **future leakage** — the model learns patterns that cannot exist at serving time. This is especially dangerous with aggregation features (rolling means, counts) that might accidentally include data from after the target event.

**Temporal feature types:**
1. **Calendar features**: hour of day, day of week, month, is_weekend, is_holiday
2. **Recency features**: time since last event per entity (days since last purchase, seconds since last login)
3. **Rolling aggregations**: mean, sum, count, std over a time window per entity (7-day average transaction amount)
4. **Trend features**: difference between recent and longer-term averages (momentum)
5. **Lag features**: target value t periods ago (essential for forecasting)

In [ ]:
# Generate synthetic transaction data with timestamps
np.random.seed(42)
n_events = 5000
start = datetime(2024, 1, 1)

events = pd.DataFrame({
    'user_id':   np.random.choice(range(100), n_events),
    'timestamp': [start + timedelta(hours=np.random.exponential(2)*i/100) for i in range(n_events)],
    'amount':    np.random.exponential(50, n_events),
    'fraud':     np.random.binomial(1, 0.03, n_events)
})
events = events.sort_values('timestamp').reset_index(drop=True)

# Extract calendar features
events['hour']        = events['timestamp'].dt.hour
events['day_of_week'] = events['timestamp'].dt.dayofweek
events['is_weekend']  = (events['day_of_week'] >= 5).astype(int)
events['month']       = events['timestamp'].dt.month

# Cyclical encoding for hour and day_of_week (important for linear models!)
# 23:00 and 00:00 should be close together — linear models can't know this with raw hour values
events['hour_sin'] = np.sin(2 * np.pi * events['hour'] / 24)
events['hour_cos'] = np.cos(2 * np.pi * events['hour'] / 24)
events['dow_sin']  = np.sin(2 * np.pi * events['day_of_week'] / 7)
events['dow_cos']  = np.cos(2 * np.pi * events['day_of_week'] / 7)

print("Calendar features added.")

# Visualize cyclical encoding
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
hours = np.arange(24)
axes[0].plot(hours, np.sin(2*np.pi*hours/24), label='sin', marker='o', markersize=4)
axes[0].plot(hours, np.cos(2*np.pi*hours/24), label='cos', marker='s', markersize=4)
axes[0].set_xlabel('Hour of day')
axes[0].set_title('Cyclical Encoding of Hour')
axes[0].legend()
axes[0].axvline(0, color='gray', linestyle='--', alpha=0.5)
axes[0].axvline(23, color='gray', linestyle='--', alpha=0.5)

axes[1].scatter(np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
                c=hours, cmap='hsv', s=100)
for h in [0, 6, 12, 18]:
    axes[1].annotate(f'{h:02d}:00',
                     (np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)),
                     fontsize=9)
axes[1].set_title('Hours as Points on Unit Circle\n(continuity preserved: 23:00 ≈ 00:00)')
axes[1].set_xlabel('sin(2π·hour/24)')
axes[1].set_ylabel('cos(2π·hour/24)')

plt.tight_layout()
plt.show()

In [ ]:
# Point-in-time correct rolling aggregations
# Key: use shift(1) to ensure features only use PAST data
events_sorted = events.sort_values(['user_id', 'timestamp'])

# Rolling 7-day stats per user — computed without future leakage
# Method: sort by user+time, then for each row, only look backward
events_sorted = events_sorted.set_index('timestamp')

def compute_rolling_features(group, window='7D'):
    # shift(1): exclude current row (point-in-time correctness)
    group = group.sort_index()
    group[f'rolling_{window}_mean_amount']  = group['amount'].shift(1).rolling(window, min_periods=1).mean()
    group[f'rolling_{window}_count']        = group['amount'].shift(1).rolling(window, min_periods=1).count()
    group[f'rolling_{window}_std_amount']   = group['amount'].shift(1).rolling(window, min_periods=1).std()
    return group

events_featured = events_sorted.groupby('user_id', group_keys=False).apply(compute_rolling_features)
events_featured = events_featured.reset_index()

print("Rolling feature example for user_id=0:")
cols = ['timestamp', 'user_id', 'amount', 'rolling_7D_mean_amount', 'rolling_7D_count']
print(events_featured[events_featured['user_id']==0][cols].head(8).to_string())

## 3. Entity-Level Aggregation Features

Many production ML problems (fraud, churn, recommendation) involve **entity-level features** — aggregations computed over all historical events for a given user, device, or merchant. These capture behavioral profiles:

- `user_lifetime_transaction_count`
- `user_avg_transaction_amount_last_30d`
- `merchant_fraud_rate_last_90d`
- `device_new_account_count_1d`

Engineering considerations:
1. **Freshness**: how stale can the feature be? Real-time (milliseconds) vs. batch (daily refreshed)
2. **Cold start**: a new user has no history — need fallback (global average, null indicator)
3. **Cardinality**: millions of users × thousands of features = huge storage requirements → feature store
4. **Temporal correctness**: when creating training labels, join features as they would have appeared at label creation time, not at query time

In [ ]:
# Demonstrate point-in-time correct entity feature join
# Scenario: predict fraud at transaction time using user profile computed up to that point

# Step 1: build entity profiles at each point in time
# Use expanding window (all history up to current row)
events_sorted2 = events.sort_values(['user_id', 'timestamp']).copy()

def compute_expanding_profile(group):
    group = group.sort_values('timestamp')
    # shift(1) ensures we don't use the current transaction
    group['user_tx_count']        = group['amount'].shift(1).expanding().count()
    group['user_avg_amount']      = group['amount'].shift(1).expanding().mean()
    group['user_fraud_rate']      = group['fraud'].shift(1).expanding().mean()
    group['user_days_since_start'] = (group['timestamp'] - group['timestamp'].min()).dt.total_seconds() / 86400
    return group

events_profiled = events_sorted2.groupby('user_id', group_keys=False).apply(compute_expanding_profile)
events_profiled[['user_tx_count', 'user_avg_amount', 'user_fraud_rate']] = \
    events_profiled[['user_tx_count', 'user_avg_amount', 'user_fraud_rate']].fillna(0)

print("Entity feature profile for user_id=5:")
cols2 = ['timestamp', 'amount', 'fraud', 'user_tx_count', 'user_avg_amount', 'user_fraud_rate']
print(events_profiled[events_profiled['user_id']==5][cols2].head(8).to_string())

In [ ]:
# Train a model with entity features vs. without
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

base_feats   = ['amount', 'hour', 'is_weekend', 'day_of_week']
entity_feats = base_feats + ['user_tx_count', 'user_avg_amount', 'user_fraud_rate']

X_all = events_profiled[entity_feats].fillna(0).values
y_all = events_profiled['fraud'].values

# Filter out NaN rows
valid = ~np.isnan(X_all).any(axis=1)
X_all = X_all[valid]
y_all = y_all[valid]

from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=False)  # Note: should use TimeSeriesSplit in production

for feat_list, label in [(base_feats, 'Base features'), (entity_feats, 'Base + entity features')]:
    n = len(feat_list)
    Xf = events_profiled[feat_list].fillna(0).values[valid]
    pipe = Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(class_weight='balanced', max_iter=500))])
    from sklearn.metrics import roc_auc_score
    scores = cross_val_score(pipe, Xf, y_all, cv=skf, scoring='roc_auc')
    print(f"{label:35s} | AUC = {scores.mean():.4f} ± {scores.std():.4f}  (n_features={n})")

## 4. Feature Store Architecture Principles

A **feature store** is a centralized system for storing, managing, and serving ML features. It solves the core engineering problems:

```
┌─────────────────────────────────────────────────────────────────────┐
│                         Feature Store                               │
│                                                                     │
│  ┌─────────────────┐    ┌──────────────┐    ┌────────────────────┐ │
│  │  Feature        │    │  Offline     │    │  Online            │ │
│  │  Registry       │    │  Store       │    │  Store             │ │
│  │  (definitions)  │    │  (S3/DWH)    │    │  (Redis/DynamoDB)  │ │
│  └─────────────────┘    └──────────────┘    └────────────────────┘ │
│         │                      │                     │             │
│         ├──── Training data ──►│                     │             │
│         │    (historical join) │                     │             │
│         └──── Serving ───────────────────────────►  │             │
│              (low-latency point lookup)              │             │
└─────────────────────────────────────────────────────────────────────┘
```

Key architectural components:
1. **Feature registry**: human-readable definitions + metadata (owner, version, SLAs)
2. **Offline store**: historical feature values for training (S3, BigQuery, Snowflake)
3. **Online store**: low-latency serving of latest feature values (Redis, DynamoDB)
4. **Materialization pipeline**: computes features and writes to both stores
5. **Point-in-time join**: ensures training labels use features from the correct historical moment

In [ ]:
# Simulate a minimal feature store
class MinimalFeatureStore:
    """Toy feature store illustrating offline/online split."""
    
    def __init__(self):
        self._definitions = {}  # feature_name -> compute_fn
        self._offline = {}       # (entity_id, timestamp) -> feature_values
        self._online  = {}       # entity_id -> latest_feature_values
    
    def register_feature(self, name, compute_fn):
        self._definitions[name] = compute_fn
    
    def materialize(self, events_df, entity_col, time_col):
        """Compute all features for all events (offline batch)"""
        for _, row in events_df.iterrows():
            entity = row[entity_col]
            ts = row[time_col]
            # Only use events BEFORE current timestamp (point-in-time)
            history = events_df[
                (events_df[entity_col] == entity) & (events_df[time_col] < ts)
            ]
            features = {name: fn(history) for name, fn in self._definitions.items()}
            self._offline[(entity, ts)] = features
            self._online[entity] = features  # latest overwrites
    
    def get_training_features(self, entity, timestamp):
        return self._offline.get((entity, timestamp), {})
    
    def get_serving_features(self, entity):
        return self._online.get(entity, {})

# Register features
fs = MinimalFeatureStore()
fs.register_feature('tx_count_7d', lambda hist: len(hist.tail(7)))
fs.register_feature('avg_amount_7d', lambda hist: hist['amount'].tail(7).mean() if len(hist) > 0 else 0)

# Materialize on small subset
small_events = events_sorted2[events_sorted2['user_id'] < 3].copy()
fs.materialize(small_events, entity_col='user_id', time_col='timestamp')

# Serving: get latest features for user_id=0
print("Online features for user_id=0:", fs.get_serving_features(0))
print("\nFeature store ensures training and serving use identical computation.")

## 5. Feature Monitoring and Drift Detection

Production features drift over time as user behavior, data sources, and business conditions change. Monitoring is essential:

- **Distribution drift**: feature distribution at serving time diverges from training distribution
- **Schema drift**: upstream data changes (new null values, renamed columns)
- **Nullity drift**: fraction of null values increases (upstream system issue)

Detection methods:
- **KL divergence / Population Stability Index (PSI)** for continuous features
- **Chi-squared test** for categorical features
- **Z-score monitoring** on summary statistics (mean, std, null rate)

In [ ]:
# Population Stability Index (PSI) — standard industry metric for feature drift
def compute_psi(expected, actual, n_bins=10):
    """PSI: < 0.1 stable; 0.1-0.2 minor shift; > 0.2 major shift"""
    bins = np.percentile(expected, np.linspace(0, 100, n_bins + 1))
    bins[0] -= 1e-9  # ensure leftmost bin includes min
    
    expected_counts = np.histogram(expected, bins=bins)[0] + 1e-9
    actual_counts   = np.histogram(actual,   bins=bins)[0] + 1e-9
    
    expected_pct = expected_counts / expected_counts.sum()
    actual_pct   = actual_counts   / actual_counts.sum()
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi

# Simulate drift scenarios
np.random.seed(42)
X_train_amount = np.random.exponential(50, 5000)

scenarios = {
    'No drift':    np.random.exponential(50, 2000),
    'Minor drift': np.random.exponential(60, 2000),   # mean shifted 20%
    'Major drift': np.random.exponential(120, 2000),  # mean doubled
    'Severe drift': np.random.normal(200, 50, 2000),  # completely different distribution
}

print("PSI scores for 'amount' feature:")
for scenario, serving_dist in scenarios.items():
    psi = compute_psi(X_train_amount, serving_dist)
    alert = '✗ MAJOR' if psi > 0.2 else ('! minor' if psi > 0.1 else 'OK stable')
    print(f"  {scenario:20s}: PSI = {psi:.4f}  [{alert}]")

print("\nPSI thresholds: < 0.1 = stable, 0.1–0.2 = investigate, > 0.2 = retrain")

## Summary

| Engineering Concern | Root Cause | Solution |
|--------------------|------------|----------|
| Training-serving skew | Different code/data for train vs. serve | Single feature definition in feature store |
| Future leakage in temporal features | Including post-event data | shift(1) before rolling windows; point-in-time joins |
| Cold start for new entities | No historical aggregations | Global fallback + missing indicator feature |
| Feature drift | Data distribution changes over time | PSI / KL divergence monitoring; retraining triggers |
| Slow serving of aggregations | Computing on-the-fly is expensive | Pre-compute and store in online store (Redis) |

| Temporal Feature Type | Example | Leakage Risk |
|----------------------|---------|---------------|
| Calendar feature | is_weekend, hour_of_day | None (derived from event timestamp only) |
| Cyclical encoding | sin/cos of hour | None |
| Rolling aggregation | 7-day avg amount | High — must use shift(1) |
| Expanding profile | lifetime tx count | High — must exclude current event |
| Lag features | last_week_sales | Medium — must align lags to prediction horizon |